# Повторный SFT: проверка адаптеров по эпохам (EXP-020)

EXP-019 не допустил ни одного адаптера SFT. Данные дополнены (8 заданий «один слайд» без поиска и 3 с поиском, портреты
с реальными именами заменены), скорость вдвое ниже (1·10⁻⁴), 6 эпох, сохранён адаптер каждой эпохи
(`configs/experiments/sft_tlite_v2.yaml`). Здесь — адаптеры всех 6 эпох при T 0.9 на тех же 25 темах validation,
по 4 ответа на тему и режим, с теми же сидами, что в EXP-019; ответы исходной T-lite при T 0.9 берутся готовыми
из EXP-019 (`runs/sft_select/20261111/base_T09`). T 0.8 не повторяется: в EXP-019 она петель не уменьшила.

Правило — то же, что в EXP-019 (`docs/decisions.md`, раздел 9): допуск против исходной T-lite при T 0.9 (правило
приёмки SFT по средним 4 ответов темы и доля вырожденных не больше, чем у исходной, плюс 5 п.п. — 10 из 200); среди
допущенных — меньшая доля вырожденных, при разнице в пределах 2 п.п. — большая награда.

Генерация — два сервера vLLM (по одному на карту, у каждого исходная модель и адаптеры шести эпох): эпохи 1–3 —
карта 0, эпохи 4–6 — карта 1. Результат — `runs/sft_select/<RUN>/`.

## 1. Серверы vLLM (терминал)

`SFT_RUN` — папка запуска SFT (`sft.ipynb`, конфиг `sft_tlite_v2.yaml`), в ней `epochs/epoch_1` … `epoch_6`.
```bash
SFT_RUN=/home/jovyan/Diplom/runs/sft/<папка запуска>
source ~/vllm085/bin/activate
cd ~/Diplom
for GPU in 0 1; do
VLLM_USE_V1=0 CUDA_VISIBLE_DEVICES=$GPU nohup python -m vllm.entrypoints.openai.api_server \
    --model t-tech/T-lite-it-2.1 --revision main --served-model-name base --dtype float16 \
    --enable-lora --max-lora-rank 32 --max-loras 6 \
    --lora-modules e1=$SFT_RUN/epochs/epoch_1 e2=$SFT_RUN/epochs/epoch_2 e3=$SFT_RUN/epochs/epoch_3 e4=$SFT_RUN/epochs/epoch_4 \
    e5=$SFT_RUN/epochs/epoch_5 e6=$SFT_RUN/epochs/epoch_6 \
    --max-model-len 25600 --gpu-memory-utilization 0.85 --generation-config vllm --port 800$GPU --seed 0 \
    > ~/vllm_select_$GPU.log 2>&1 &
done
grep -h "startup complete" ~/vllm_select_*.log   # готовы оба — две строки
```
Остановить оба: `pkill -f vllm.entrypoints.openai.api_server`.

In [1]:
import json, os, statistics, sys, threading, time, traceback
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))  # если пакет не установлен через pip install -e .
from transformers import AutoTokenizer

from presentation_rl.environment import run_episode
from presentation_rl.evaluation import degeneration, evaluate_folder, paired, reward_spread, topic_means, write_csv
from presentation_rl.rewards import load_key_points
from presentation_rl.tasks import load_tasks
from presentation_rl.tools.cache import WikiCache
from presentation_rl.tools.wiki import WikiClient
from presentation_rl.tools.wiki_tools import WikiTools
from presentation_rl.training.common import LockedTokenizer, episode_prompt, mode_task, stop_ids, vllm_generator
from presentation_rl.training.vllm_client import VLLMClient

os.environ["WIKI_PROXY"] = ""  # Википедия открыта в пространстве; туннель — docs/server.md
RUN = "20261112_exp020"
OUT = ROOT / "runs" / "sft_select" / RUN
MODEL_ID = "t-tech/T-lite-it-2.1"
BASE = ("base_T09", ROOT / "runs" / "sft_select" / "20261111" / "base_T09")  # готовые ответы исходной (EXP-019)
MODELS = ("e1", "e2", "e3", "e4", "e5", "e6")  # адаптеры эпох (--lora-modules)
TEMPERATURES = (0.9,)
SERVERS = {"e1": "http://127.0.0.1:8000", "e2": "http://127.0.0.1:8000", "e3": "http://127.0.0.1:8000",
           "e4": "http://127.0.0.1:8001", "e5": "http://127.0.0.1:8001", "e6": "http://127.0.0.1:8001"}
SAMPLES, MAX_NEW_TOKENS, MAX_CONTEXT, WORKERS = 4, 8000, 25000, 48
MODES = ("search", "plain")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision="main")
tasks = {t.id: t for t in load_tasks(ROOT / "data" / "tasks" / "tasks.json", split="validation")}
points = load_key_points(ROOT / "data" / "tasks" / "key_points" / "key_points.json")
IDS = sorted(tasks)
EPISODES = [(mode, key, s) for mode in MODES for key in IDS for s in range(SAMPLES)]
CONFIGS = [(f"{m}_T{round(t * 10):02d}", m, t) for t in TEMPERATURES for m in MODELS]
cache = WikiCache(ROOT / "data" / "sources" / "wiki_cache",
                  WikiClient(contact=os.environ.get("WIKI_CONTACT", "https://github.com/doshik722")))
print(f"тем {len(IDS)}, эпизодов на конфигурацию {len(EPISODES)}, конфигураций {len(CONFIGS)}")
for url in sorted(set(SERVERS.values())):
    print(url, VLLMClient(url).models())

mkdir -p failed for path /home/jovyan/.config/matplotlib: [Errno 20] Not a directory: '/home/jovyan/.config/matplotlib'
Matplotlib created a temporary cache directory at /tmp/matplotlib-us8qrhda because there was an issue with the default path (/home/jovyan/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


тем 25, эпизодов на конфигурацию 200, конфигураций 6
http://127.0.0.1:8000 ['base', 'e1', 'e2', 'e3', 'e4', 'e5', 'e6']
http://127.0.0.1:8001 ['base', 'e1', 'e2', 'e3', 'e4', 'e5', 'e6']


## 2. Генерация

Каждый сервер — свои три эпохи, эпизоды всех трёх идут вперемешку (`WORKERS` одновременно; пакеты собирает
vLLM). Эпизод сохраняется сразу: `.json` (конец, токены модели, вызовы, ошибка, время), `.txt` (эпизод после промпта),
`_answer.txt`. При перезапуске готовые эпизоды пропускаются, эпизоды с ошибкой повторяются.

In [ ]:
safe = LockedTokenizer(tokenizer)
prompt = episode_prompt(tokenizer)
PROMPTS = {(mode, key): prompt(mode_task(tasks[key], mode).request) for mode in MODES for key in IDS}  # основной поток
STOPS = stop_ids(tokenizer)


def done(folder, stem):
    path = folder / f"{stem}.json"
    return path.exists() and not json.loads(path.read_text(encoding="utf-8")).get("error")


def one_episode(client, name, model, temperature, mode, key, sample):
    folder, stem = OUT / name, f"{mode}_{key}_{sample}"
    row, start = {"mode": mode, "id": key, "sample": sample}, time.monotonic()
    try:
        prompt_ids = PROMPTS[(mode, key)]
        generate = vllm_generator(client, model, temperature, STOPS, ("sft_select", mode, key, sample))
        episode = run_episode(prompt_ids, safe, generate, WikiTools(cache) if mode == "search" else None,
                              max_new_tokens=MAX_NEW_TOKENS, max_context=MAX_CONTEXT)
        (folder / f"{stem}.txt").write_text(safe.decode(episode.ids[len(prompt_ids):], skip_special_tokens=False),
                                            encoding="utf-8")
        (folder / f"{stem}_answer.txt").write_text(episode.answer, encoding="utf-8")
        row.update(finish=episode.finish, generated_tokens=episode.generated_tokens, calls=len(episode.calls),
                   truncated_outputs=episode.truncated_outputs, error="")
    except Exception:
        row.update(error=traceback.format_exc()[-1500:])
    row["seconds"] = round(time.monotonic() - start)
    (folder / f"{stem}.json").write_text(json.dumps(row, ensure_ascii=False, indent=1), encoding="utf-8")
    return name, row


def run_server(url):
    client = VLLMClient(url)
    temperature = TEMPERATURES[0]
    jobs = []
    for name, model, t in CONFIGS:
        if SERVERS[model] == url:
            (OUT / name).mkdir(parents=True, exist_ok=True)
            jobs += [(name, model, mode, key, s) for mode, key, s in EPISODES if not done(OUT / name, f"{mode}_{key}_{s}")]
    jobs.sort(key=lambda j: (j[4], j[3], j[2], j[0]))  # конфигурации вперемешку: сравнимый ход у всех
    counts, errors, start = {}, 0, time.monotonic()
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        futures = [pool.submit(one_episode, client, name, model, temperature, mode, key, s)
                   for name, model, mode, key, s in jobs]
        for k, future in enumerate(futures, 1):
            name, row = future.result()
            counts[name] = counts.get(name, 0) + 1
            errors += bool(row["error"])
            if k % 50 == 0 or k == len(futures):
                print(f"{url}: {k}/{len(futures)} за {(time.monotonic() - start) / 60:.0f} мин, ошибок {errors}",
                      flush=True)


threads = [threading.Thread(target=run_server, args=(url,)) for url in sorted(set(SERVERS.values()))]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()

http://127.0.0.1:8001: 50/600 за 66 мин, ошибок 0
http://127.0.0.1:8001: 100/600 за 77 мин, ошибок 0
http://127.0.0.1:8001: 150/600 за 111 мин, ошибок 0
http://127.0.0.1:8000: 50/600 за 123 мин, ошибок 0
http://127.0.0.1:8001: 200/600 за 124 мин, ошибок 0
http://127.0.0.1:8001: 250/600 за 174 мин, ошибок 0
http://127.0.0.1:8000: 100/600 за 177 мин, ошибок 0
http://127.0.0.1:8001: 300/600 за 188 мин, ошибок 0
http://127.0.0.1:8000: 150/600 за 210 мин, ошибок 0
http://127.0.0.1:8001: 350/600 за 225 мин, ошибок 0
http://127.0.0.1:8001: 400/600 за 250 мин, ошибок 0
http://127.0.0.1:8000: 200/600 за 254 мин, ошибок 0
http://127.0.0.1:8001: 450/600 за 287 мин, ошибок 0
http://127.0.0.1:8000: 250/600 за 302 мин, ошибок 0
http://127.0.0.1:8001: 500/600 за 307 мин, ошибок 0
http://127.0.0.1:8001: 550/600 за 319 мин, ошибок 0
http://127.0.0.1:8001: 600/600 за 327 мин, ошибок 0
http://127.0.0.1:8000: 300/600 за 350 мин, ошибок 0
http://127.0.0.1:8000: 350/600 за 410 мин, ошибок 0


In [ ]:
import subprocess, time
subprocess.run(["pkill", "-f", "vllm.entrypoints.openai.api_server"])  # серверы больше не нужны — карта 1 для bge-m3
time.sleep(30)

## 3. Оценка

После генерации: остановить серверы (`pkill -f vllm.entrypoints.openai.api_server`) — bge-m3 занимает карту 1.
Ответы исходной T-lite оцениваются заново тем же кодом (награда та же), из папки EXP-019.

In [ ]:
from presentation_rl.rewards import Lemmatizer, SentenceEmbedder

embedder, lemmatizer = SentenceEmbedder("BAAI/bge-m3", device="cuda:1"), Lemmatizer()
rows = evaluate_folder(BASE[1], BASE[0], tasks, points, embedder, lemmatizer, build_pptx=False)
print(f"{BASE[0]} (EXP-019): эпизодов {len(rows)}")
for name, model, t in CONFIGS:
    part = evaluate_folder(OUT / name, name, tasks, points, embedder, lemmatizer, build_pptx=False)
    print(f"{name}: эпизодов {len(part)}, ошибок {sum(bool(r.get('error')) for r in part)}")
    rows += part
write_csv(OUT / "episodes.csv", rows)

## 4. Допуск и выбор

Каждая эпоха — против исходной T-lite при T 0.9 (ответы EXP-019).

In [ ]:
METRICS = ["parse_rate", "slides", "in_range", "conclusion", "stop", "searched", "calls", "sources_ok", "sources_bad",
           "images", "image_words", "images_related", "generated_tokens", "degenerate", "loop_8gram", "reward_total",
           "reward_coverage", "reward_keys", "reward_length", "reward_visual", "reward_format"]


def share(name, key="degenerate", mode=None):
    part = [r for r in rows if r["model"] == name and not r.get("error") and (mode is None or r["mode"] == mode)]
    return statistics.mean(bool(r[key]) for r in part) if part else None


def mean_reward(name):  # средняя по темам награда, среднее двух режимов
    means = [topic_means(rows, name, "reward_total", mode) for mode in MODES]
    return statistics.mean(statistics.mean(m.values()) for m in means if m) if any(means) else None


def admission(name, base):
    test = lambda metric, mode: paired(rows, metric, mode, base, name)
    worse = lambda t: t is not None and t["high"] < 0
    better = lambda t: t is not None and t["low"] > 0
    bad_sources = test("sources_bad", "search")
    spread_base, spread = reward_spread(rows, base), reward_spread(rows, name)
    checks = {
        "ok_parse_rate": not any(worse(test("parse_rate", m)) for m in MODES),
        "ok_sources_bad": not (bad_sources is not None and bad_sources["low"] > 0),
        "ok_better_one": any(better(test(metric, mode)) for metric, mode in
                          (("conclusion", "plain"), ("conclusion", "search"), ("in_range", "plain"),
                           ("in_range", "search"), ("searched", "search"))),
        "ok_reward": not any(worse(test("reward_total", m)) for m in MODES),
        "ok_spread": spread is not None and spread_base is not None and spread >= spread_base / 2,
        "ok_degeneration": share(name) <= share(base) + 0.05,
    }
    return checks


table, selection = [], []
for name, model, t in [(BASE[0], "base", 0.9)] + CONFIGS:
    base = BASE[0]
    row = {"config": name, "degenerate": share(name), "degenerate_search": share(name, mode="search"),
           "degenerate_plain": share(name, mode="plain"), "unfinished": share(name, "unfinished"),
           "reward": mean_reward(name), "spread": reward_spread(rows, name),
           "tokens_median": statistics.median(r["generated_tokens"] for r in rows if r["model"] == name and not r.get("error"))}
    if name != base:
        checks = admission(name, base)
        row.update(checks, admitted=all(checks.values()))
        table += [{"config": name, "against": base, **c} for mode in MODES for metric in METRICS
                  if (c := paired(rows, metric, mode, base, name))]
    selection.append(row)
write_csv(OUT / "comparison.csv", table)
write_csv(OUT / "selection.csv", selection)
for r in selection:
    print("  ".join(f"{k} {v:.3f}" if isinstance(v, float) else f"{k} {v}" for k, v in r.items()))

admitted = [r for r in selection if r.get("admitted")]
if admitted:
    best = min(r["degenerate"] for r in admitted)
    close = [r for r in admitted if r["degenerate"] <= best + 0.02]
    print("Выбор по правилу:", max(close, key=lambda r: r["reward"])["config"], "| допущены:", [r["config"] for r in admitted])
else:
    print("Не допущена ни одна конфигурация — решение после разбора (правило, п. «если не допущена ни одна»)")